# Fit on standardized features, report in raw units

A price model for used cars uses three features: age in years, distance driven in
kilometers and engine size in liters. The distance column is tens of thousands of times
larger than the others, so gradient descent on the raw columns faces a condition number in
the billions. Standardizing makes the fit take a few dozen steps, but its weights belong to
the standardized columns, and the people using the model want dollars per year, per
kilometer and per liter. Here you fit on standardized features and report the model in raw
units.

**How to work through it.** Each part names one function to write. Fill in its
body where it says `TODO` and delete the `raise NotImplementedError` line, then run
the check cell below it. A correct function prints `Part N passed`. Otherwise the
check stops and says what went wrong.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 160
age = rng.uniform(1, 15, n)                                  # years
km = np.abs(9000 * age + rng.normal(0, 30000, n))            # kilometers driven
engine = rng.uniform(1.0, 3.5, n)                            # liters
X = np.column_stack([age, km, engine])
y = 32000 - 1100 * age - 0.04 * km + 3500 * engine + rng.normal(0, 1500, n)   # price in dollars

X_new = np.array([[3.0, 41000.0, 1.6],                       # three cars to price later
                  [9.0, 95000.0, 2.0],
                  [14.0, 150000.0, 3.0]])

for label, M in (("raw, centered", X - X.mean(axis=0)),
                 ("standardized", (X - X.mean(axis=0)) / X.std(axis=0))):
    lam = np.linalg.eigvalsh(2 / n * M.T @ M)
    print(f"{label:14} condition number of the MSE Hessian: {lam[-1] / lam[0]:.3g}")

## Part 1 (coding task): Fit on standardized columns, answer in raw units

Write `fit_scaled_gd(X, y, steps=100)`. It returns a pair `(w, b)`, a weight per raw
column and an intercept, so that `X @ w + b` is the fitted price.

1. Standardize each column of `X` with its mean and its standard deviation
   (`X.std(axis=0)`), and center `y` by subtracting its mean. The standardized fit then
   needs no intercept.
2. Run `steps` steps of gradient descent on the mean squared error
   $\tfrac1n\lVert \mathbf{y}_c - Z\theta \rVert^2$ from $\theta = \mathbf{0}$, with the
   best fixed step $2/(\lambda_{\min} + \lambda_{\max})$ of its Hessian
   $\tfrac2n Z^\top Z$.
3. Turn $\theta$ into `w` and `b` that give exactly the same predictions on raw inputs.

Work out step 3 yourself: write one standardized feature in terms of the raw one and
collect the terms.

In [ ]:
def fit_scaled_gd(X, y, steps=100):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    # TODO: standardize, run gradient descent, then convert theta to raw-unit (w, b).
    raise NotImplementedError

In [ ]:
w, b = fit_scaled_gd(X, y)
w = np.asarray(w, dtype=float)
b = float(b)
assert w.shape == (3,), f"one weight per feature: expected shape (3,), got {w.shape}"
A = np.column_stack([np.ones(n), X])
best = np.linalg.lstsq(A, y, rcond=None)[0]          # intercept, then the three weights
sd = X.std(axis=0)
if not np.allclose(w, best[1:], rtol=1e-6, atol=0):
    assert not np.allclose(w, best[1:] * sd, rtol=1e-3), "these are the standardized weights theta: convert them to weights per raw unit"
    assert not np.allclose(w, best[1:] * sd ** 2, rtol=1e-3), "the weights are multiplied by the standard deviation where they should be divided by it"
    raise AssertionError(f"the weights {w} are not the least-squares weights {best[1:]}: check the standardizing, the step size and the update")
assert not np.isclose(b, y.mean()), "y.mean() is the intercept only when every raw feature has mean 0: the centering of X has to move into b"
assert np.isclose(b, best[0], rtol=1e-6), f"the intercept {b:.2f} should be {best[0]:.2f}"
assert np.allclose(X_new @ w + b, X_new @ best[1:] + best[0], rtol=1e-6), "the predictions on new cars do not match the least-squares fit"
w2, b2 = fit_scaled_gd(X[:, [2, 0]], y)
best2 = np.linalg.lstsq(np.column_stack([np.ones(n), X[:, [2, 0]]]), y, rcond=None)[0]
assert np.allclose(np.r_[b2, w2], best2, rtol=1e-6), "on the two columns engine and age the fit is wrong: the function should work for any X"
print("price of the three new cars:", np.round(X_new @ w + b))
print("dollars per year, per km, per liter:", np.round(w, 3))
print("Part 1 passed")

## Solution

### Part 1: Fit on standardized columns, answer in raw units

In [ ]:
def fit_scaled_gd(X, y, steps=100):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    mean, std = X.mean(axis=0), X.std(axis=0)
    Z = (X - mean) / std
    y_c = y - y.mean()
    lam = np.linalg.eigvalsh(2 / n * Z.T @ Z)
    eta = 2 / (lam[0] + lam[-1])                 # best fixed step for this bowl
    theta = np.zeros(d)
    while steps > 0:
        theta = theta + eta * 2 / n * Z.T @ (y_c - Z @ theta)
        steps -= 1
    w = theta / std                      # weight per raw unit
    b = y.mean() - w @ mean              # the centering, moved into the intercept
    return w, b

**Why it works.** Write $m_k$ and $s_k$ for the mean and standard deviation of raw feature $k$. The
standardized model predicts

$$
\bar y + \sum_k \theta_k \, \frac{x_k - m_k}{s_k}
= \sum_k \frac{\theta_k}{s_k}\, x_k \;+\; \Big(\bar y - \sum_k \frac{\theta_k\, m_k}{s_k}\Big) .
$$

So the raw weight of feature $k$ is $\theta_k/s_k$, and the intercept is $\bar y$ minus
the raw weights dotted with the means. Both models make the same predictions, so the
least-squares fit in one set of units is the least-squares fit in the other. After
standardizing, the condition number is about 7.6, so the best step shrinks the error by
$(\kappa - 1)/(\kappa + 1) \approx 0.77$ per step and 100 steps leave about $10^{-12}$ of
it. On the centered raw columns the condition number is about $3 \times 10^9$, and the same
100 steps would leave the age weight near $-0.001$ dollars per year instead of $-1103$. Centering `y` is optional
here: the standardized columns have mean 0, so $Z^\top\mathbf{y} = Z^\top\mathbf{y}_c$.